# Preparar el VPS

Configuración inicial, claves SSH, firewall

## Introducción

Un VPS (Virtual Private Server) es tu propio servidor en la nube donde puedes desplegar aplicaciones Python/FastAPI. En esta lección aprenderás a configurar un servidor Ubuntu desde el primer acceso hasta tenerlo listo para producción: usuarios, firewall, Docker y seguridad básica.

### Objetivos de Aprendizaje

- Conectar al servidor por primera vez vía SSH
- Crear un usuario no-root con permisos sudo
- Configurar el firewall UFW para proteger el servidor
- Instalar Docker y Docker Compose en Ubuntu/Debian
- Instalar y configurar nginx como servidor web
- Aplicar medidas básicas de seguridad (fail2ban, SSH keys)

## Primer Acceso por SSH

> SSH (Secure Shell) es el protocolo estándar para conectarse a servidores remotos de forma segura. Tu proveedor de VPS te dará una IP y contraseña root (o una clave SSH). Lo primero es acceder y actualizar el sistema.

In [ ]:
# Desde tu máquina local (terminal):
# ssh root@203.0.113.10

# Copiar tu clave SSH pública al servidor (más seguro que contraseñas)
# ssh-copy-id root@203.0.113.10

# Una vez conectado, actualizar el sistema
comandos_iniciales = [
    "apt update",
    "apt upgrade -y",
    "apt install -y curl wget git htop unzip",
]

print("Sistema actualizado y herramientas básicas instaladas")

## Crear Usuario No-Root

> Trabajar siempre como root es peligroso. Un error puede dañar el sistema completo. La práctica segura es crear un usuario dedicado para despliegues con permisos sudo limitados.

In [ ]:
pasos_usuario = """
# 1. Crear el usuario 'deploy'
adduser deploy

# 2. Agregar al grupo sudo
usermod -aG sudo deploy

# 3. Verificar que funciona
su - deploy
sudo whoami  # debe imprimir: root

# 4. Copiar claves SSH al nuevo usuario
# ssh-copy-id deploy@203.0.113.10
"""

print(pasos_usuario)

grupos_esperados = ["sudo", "docker"]
print("Grupos que debe tener el usuario deploy:")
for grupo in grupos_esperados:
    print(f"  - {grupo}")

## Configurar Firewall UFW

> UFW (Uncomplicated Firewall) es la forma más sencilla de gestionar iptables en Ubuntu. Solo debes abrir los puertos que realmente necesitas: SSH (22), HTTP (80) y HTTPS (443).

In [ ]:
pasos_ufw = """
# Ver estado actual
ufw status

# Permitir puertos esenciales ANTES de habilitar
ufw allow 22/tcp    # SSH
ufw allow 80/tcp    # HTTP
ufw allow 443/tcp   # HTTPS

# Habilitar el firewall
ufw enable

# Ver reglas activas
ufw status verbose
"""

print(pasos_ufw)

reglas_requeridas = [22, 80, 443]
print("Puertos que deben estar abiertos:")
for puerto in reglas_requeridas:
    print(f"  Puerto {puerto}: PERMITIDO")

## Instalar Docker y Docker Compose

> Docker permite empaquetar tu aplicación con todas sus dependencias. En lugar de instalar Python, librerías y configurar el entorno manualmente, simplemente corres un contenedor.

In [ ]:
script_docker = """
# 1. Instalar dependencias
apt update
apt install -y ca-certificates curl gnupg lsb-release

# 2. Agregar clave GPG oficial de Docker
mkdir -p /etc/apt/keyrings
curl -fsSL https://download.docker.com/linux/ubuntu/gpg \\
    | gpg --dearmor -o /etc/apt/keyrings/docker.gpg

# 3. Agregar repositorio Docker
echo "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/docker.gpg] \\
    https://download.docker.com/linux/ubuntu \\
    $(lsb_release -cs) stable" \\
    | tee /etc/apt/sources.list.d/docker.list > /dev/null

# 4. Instalar Docker Engine
apt update
apt install -y docker-ce docker-ce-cli containerd.io docker-compose-plugin

# 5. Agregar usuario al grupo docker
usermod -aG docker deploy

# 6. Verificar instalación
docker --version
docker compose version
"""

print(script_docker)

print("Verificando Docker...")
print("  docker --version: OK")
print("  docker compose version: OK")

## Instalar Nginx y Configurar Hostname

> Nginx actúa como puerta de entrada a tu servidor, redirigiendo el tráfico web a tus contenedores Docker. También sirve archivos estáticos muy eficientemente. El hostname le da un nombre legible al servidor.

In [ ]:
configuracion_servidor = """
# Instalar nginx
apt install -y nginx

# Iniciar y habilitar para que arranque al reiniciar
systemctl start nginx
systemctl enable nginx
systemctl status nginx

# Configurar hostname del servidor
hostnamectl set-hostname mi-servidor-produccion

# Verificar
hostname
hostnamectl status

# Comprobar nginx funcionando
curl http://localhost
"""

print(configuracion_servidor)

estructura_nginx = {
    "/etc/nginx/nginx.conf": "Configuración principal",
    "/etc/nginx/sites-available/": "Sitios disponibles (configs)",
    "/etc/nginx/sites-enabled/": "Sitios activos (symlinks)",
    "/var/www/html/": "Archivos web por defecto",
    "/var/log/nginx/": "Logs de acceso y errores",
}

print("Estructura de directorios de Nginx:")
for ruta, descripcion in estructura_nginx.items():
    print(f"  {ruta}: {descripcion}")

## Seguridad Básica: fail2ban y SSH

> fail2ban bloquea automáticamente IPs que intentan ataques de fuerza bruta. Configurar SSH para usar solo claves (sin contraseñas) y deshabilitar el login de root son medidas esenciales de seguridad.

In [ ]:
seguridad_ssh = """
# 1. Instalar fail2ban
apt install -y fail2ban

# Crear configuración local
cp /etc/fail2ban/jail.conf /etc/fail2ban/jail.local

# Configuración básica en /etc/fail2ban/jail.local:
# [sshd]
# enabled = true
# port = ssh
# maxretry = 5
# bantime = 3600

systemctl enable fail2ban
systemctl start fail2ban

# 2. Deshabilitar login con contraseña y root en SSH
# Editar /etc/ssh/sshd_config:
# PermitRootLogin no
# PasswordAuthentication no
# PubkeyAuthentication yes

# IMPORTANTE: Asegúrate de tener tu clave SSH configurada
# ANTES de deshabilitar contraseñas, o te quedarás fuera!

# Reiniciar SSH
systemctl restart sshd
"""

print(seguridad_ssh)

medidas = [
    ("fail2ban activo", "Bloquea ataques de fuerza bruta"),
    ("Root login deshabilitado", "Solo usuarios normales pueden entrar"),
    ("Solo SSH keys", "Sin contraseñas = sin ataques de diccionario"),
    ("UFW activo", "Solo puertos 22, 80, 443 abiertos"),
    ("Usuario no-root", "Daños limitados ante un error"),
]

print("\nMedidas de seguridad aplicadas:")
for medida, descripcion in medidas:
    print(f"  ✓ {medida}: {descripcion}")

## Script de Setup Completo

Un script que documenta y ejecuta todos los pasos de configuración del VPS en orden correcto.

In [ ]:
print("=== SETUP VPS UBUNTU ===")
print("Pasos a ejecutar:")
pasos_display = [
    "1. Actualizar sistema (apt update/upgrade)",
    "2. Instalar curl, wget, git, htop, fail2ban, nginx",
    "3. Configurar UFW (puertos 22, 80, 443)",
    "4. Activar UFW",
    "5. Iniciar nginx",
    "6. Iniciar fail2ban",
]
for paso in pasos_display:
    print(f"  {paso}")

pasos = [
    ("apt update && apt upgrade -y", "Actualizar sistema"),
    ("apt install -y curl wget git htop fail2ban nginx", "Instalar paquetes base"),
    ("ufw allow ssh && ufw allow http && ufw allow https", "Configurar firewall"),
    ("ufw --force enable", "Activar firewall"),
    ("systemctl enable nginx && systemctl start nginx", "Iniciar nginx"),
    ("systemctl enable fail2ban && systemctl start fail2ban", "Iniciar fail2ban"),
]

print("\nSecuencia de comandos:")
for i, (comando, desc) in enumerate(pasos, 1):
    print(f"\n{i}. {desc}")
    print(f"   $ {comando}")

## Tips y Mejores Prácticas

> NUNCA actives el firewall UFW sin primero permitir el puerto 22 (SSH). Si lo haces, te quedarás bloqueado fuera del servidor sin forma de entrar.

> Antes de deshabilitar el login por contraseña en SSH, abre una segunda sesión SSH para verificar que tu clave funciona. Así tienes un "plan B" si algo falla.

> El comando ssh-copy-id copia tu clave pública (~/.ssh/id_rsa.pub) al servidor automáticamente. Si no tienes claves SSH, créalas con: ssh-keygen -t ed25519

> Usa hostnamectl set-hostname nombre-descriptivo para identificar fácilmente tus servidores. Especialmente útil cuando tienes múltiples VPS.

> Después de agregar un usuario al grupo docker (usermod -aG docker deploy), el usuario debe cerrar sesión y volver a entrar para que el cambio tenga efecto.

> fail2ban por defecto protege SSH: después de 5 intentos fallidos de login, bloquea la IP por 10 minutos. Puedes ver los bans con: fail2ban-client status sshd

## Errores Comunes

### Activar UFW sin permitir SSH primero

¿Por qué ocurre?
- UFW denegará todas las conexiones entrantes incluyendo tu SSH activo, bloqueándote del servidor.

Solución
- Siempre ejecuta ufw allow ssh antes de ufw enable. Si ya te bloqueaste, necesitarás la consola de emergencia de tu proveedor VPS.

### Deshabilitar contraseñas SSH sin tener clave SSH configurada

¿Por qué ocurre?
- Te quedarás sin forma de acceder al servidor. PasswordAuthentication no + sin clave = acceso imposible.

Solución
- Primero copia tu clave con ssh-copy-id, luego verifica que puedes entrar con clave, y ENTONCES deshabilita las contraseñas.

### Usar siempre el usuario root para todo

¿Por qué ocurre?
- Un error de comandos como rm -rf en el directorio equivocado puede destruir el sistema operativo completo.

Solución
- Crea un usuario no-root con sudo. Usa sudo solo cuando sea necesario, no de forma permanente.

### No actualizar el sistema después de la instalación inicial

¿Por qué ocurre?
- Los servidores recién creados pueden tener vulnerabilidades conocidas en paquetes desactualizados.

Solución
- Ejecuta apt update && apt upgrade -y inmediatamente después del primer acceso.

### Abrir todos los puertos en UFW "para no tener problemas"

¿Por qué ocurre?
- Expone servicios internos (bases de datos, APIs de administración) a internet.

Solución
- Principio de mínimo privilegio: abre solo los puertos que realmente necesitas (22, 80, 443 para la mayoría de apps web).